# 01. 전문가 중요도 측정 (A100 80GB 또는 G4, 약 1시간 / 약 8~10유닛)
한국어 위키와 영어 위키를 각각 모델에 흘려 보내며, 층마다 32개 전문가가
- 얼마나 자주 뽑히는지 (빈도, `freq`)
- 뽑혔을 때 출력에 얼마나 크게 기여하는지 (REAP 점수, `reap`)

를 기록합니다. 결과는 작은 JSON 파일(`results/calib/stats-ko.json`, `stats-en.json`)이고 드라이브에 저장됩니다.

원본 MXFP4를 불러오는 데 문제가 생기면 `--model unsloth/gpt-oss-20b-BF16` 을 붙이세요. 값이 같은 BF16 버전입니다.

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 45, "런타임 > 런타임 유형 변경에서 A100 (80GB) 또는 G4 를 고르세요"

In [ ]:
!python scripts/01_calibrate.py --langs ko en
push_results()

## 한국어와 영어는 같은 전문가를 쓸까? (보고서용 그림)

In [ ]:
import json, matplotlib.pyplot as plt
from cfinetune.observe import select_experts
ko = json.load(open("results/calib/stats-ko.json")); en = json.load(open("results/calib/stats-en.json"))
for k in (24, 16):
    sk, se = select_experts(ko, "reap", k), select_experts(en, "reap", k)
    overlap = [len(set(sk[i]) & set(se[i])) / k for i in sk]
    plt.plot(overlap, marker="o", label=f"keep {k}")
plt.xlabel("layer"); plt.ylabel("ko/en overlap of kept experts"); plt.ylim(0, 1.05); plt.grid(alpha=.3); plt.legend()
plt.savefig("results/fig_expert_overlap.png", dpi=150); plt.show()
push_results()